# Chapter 3 — Keyword and semantic retrieval for RAG

Companion code for **Chapter 3** of *Build an Advanced RAG Application (From Scratch)*.

We build the two foundational retrievers of RAG from scratch and run them on the same
corpus of hotel reviews:

1. **Keyword search** — an inverted index and TF-IDF scoring. Fast, no model, precise on exact strings.
2. **Semantic search** — sentence embeddings from `nomic-embed-text-v1.5` and cosine similarity. Matches meaning, not words.

Along the way we see why documents must be *chunked* before encoding: the hard token limit,
and the softer loss of precision when one vector has to stand for too many ideas.

> **Prerequisites:** `sentence-transformers`, `transformers`, `nltk`, `numpy`, `torch`.
> Run the install cell once, then restart the kernel. No API keys are needed; the models
> download from HuggingFace on first run.

## 0. Setup

In [1]:
# Run once; restart the kernel afterwards.
!pip install sentence-transformers transformers nltk numpy torch --quiet

## 1. Imports and device selection

We pick **CUDA → MPS → CPU** automatically so the same code runs on any machine.
Everything in this chapter runs comfortably on a laptop CPU.

In [2]:
import math

import nltk
import numpy as np
import torch
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
from transformers import BertTokenizer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Using device: {DEVICE}")

Using device: mps


## 2. The Travelle review corpus

Eight hotel reviews, the same kind of text Travelle searches. The first three are the reviews
from Chapter 2's encoder demo, carried forward so the numbers line up. Small enough to read
every score by hand, varied enough to expose every behavior we care about.

In [3]:
REVIEWS = [
    "We slept with the windows open and never heard a thing, right in the middle of Paris.",
    "A calm little place far from the traffic, with birds singing in the courtyard.",
    "The walls were paper-thin and we heard every footstep in the hallway.",
    "Hotel du Marais was spotless and the staff upgraded our room for free.",
    "Our top-floor room looked straight out at the Eiffel Tower, lit up every night.",
    "Right on the Rue de Rivoli, a five-minute walk to the museums and the Tuileries gardens.",
    "The bar by the pool blasted music until well past midnight.",
    "Dated rooms, but the location near the airport is handy for an early flight.",
]

QUERIES = [
    "somewhere peaceful where I can sleep",  # semantic win: paraphrase, keyword all-zeros
    "Hotel du Marais",                       # keyword win: exact name
    "a short stroll from the Louvre",        # semantic win: keyword buries the right doc
    "a hotel not near the airport",          # both fail: negation
    "a room with a view of the Eiffel Tower",  # landmark, richer query
]

print(f"{len(REVIEWS)} reviews loaded.")

8 reviews loaded.


## 3. Keyword search with an inverted index and TF-IDF

### 3.1 Building an inverted index

An inverted index maps each term to the list of documents (a *posting list*) that contain it.
It is built once, offline, and lets a query touch only the documents that share a word with it.

In [4]:
def build_inverted_index(documents):
    tokenized_docs = [word_tokenize(doc) for doc in documents]     # keep original casing for counts
    index = {}
    for doc_id, tokens in enumerate(tokenized_docs):
        for token in set(t.lower() for t in tokens):               # lowercase only the index keys
            index.setdefault(token, []).append(doc_id)
    return tokenized_docs, index


tokenized_docs, inverted_index = build_inverted_index(REVIEWS)
print(f"vocabulary size: {len(inverted_index)} unique tokens")
for token in ["quiet", "hotel", "airport"]:
    print(f"  '{token}' -> docs {inverted_index.get(token, [])}")

vocabulary size: 84 unique tokens
  'quiet' -> docs []
  'hotel' -> docs [3]
  'airport' -> docs [7]


Note the first line: **`'quiet' -> docs []`**. The word appears in none of the reviews, even
though two of them describe exactly that. The inverted index can only find words that were
literally written — the hinge the whole chapter turns on.

### 3.2 TF-IDF scoring

TF-IDF ranks a document high for a term that appears **often here** (term frequency) but
**rarely across the collection** (inverse document frequency). A rare, specific word outscores
a ubiquitous one by more than an order of magnitude.

In [5]:
def compute_tfidf(token, doc_id, tokenized_docs, index):
    tf = [t.lower() for t in tokenized_docs[doc_id]].count(token.lower())
    df = len(index.get(token.lower(), []))
    idf = math.log(len(tokenized_docs) / (df + 1))   # smoothed; +1 avoids div-by-zero
    return tf * idf


def keyword_search(query, documents, tokenized_docs, index):
    query_tokens = word_tokenize(query.lower())
    scores = {doc_id: 0.0 for doc_id in range(len(documents))}
    for token in query_tokens:
        for doc_id in index.get(token, []):          # only docs on a posting list can score
            scores[doc_id] += compute_tfidf(token, doc_id, tokenized_docs, index)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# Work one weight by hand to check the formula.
print("weight of 'airport' in review 7:",
      round(compute_tfidf("airport", 7, tokenized_docs, inverted_index), 3))
print("weight of 'the' in review 7    :",
      round(compute_tfidf("the", 7, tokenized_docs, inverted_index), 3))

weight of 'airport' in review 7: 1.386
weight of 'the' in review 7    : -0.236


### 3.3 Querying keyword search

In [6]:
def show(query, results):
    print(f"query: '{query}'")
    for doc_id, score in results[:5]:
        mark = "*" if score > 0 else " "
        print(f"  {mark} [{doc_id}] {score:.4f}  {REVIEWS[doc_id]}")
    print()


for q in QUERIES:
    show(q, keyword_search(q, REVIEWS, tokenized_docs, inverted_index))

query: 'somewhere peaceful where I can sleep'
    [0] 0.0000  We slept with the windows open and never heard a thing, right in the middle of Paris.
    [1] 0.0000  A calm little place far from the traffic, with birds singing in the courtyard.
    [2] 0.0000  The walls were paper-thin and we heard every footstep in the hallway.
    [3] 0.0000  Hotel du Marais was spotless and the staff upgraded our room for free.
    [4] 0.0000  Our top-floor room looked straight out at the Eiffel Tower, lit up every night.

query: 'Hotel du Marais'
  * [3] 4.1589  Hotel du Marais was spotless and the staff upgraded our room for free.
    [0] 0.0000  We slept with the windows open and never heard a thing, right in the middle of Paris.
    [1] 0.0000  A calm little place far from the traffic, with birds singing in the courtyard.
    [2] 0.0000  The walls were paper-thin and we heard every footstep in the hallway.
    [4] 0.0000  Our top-floor room looked straight out at the Eiffel Tower, lit up every nig

Read the failures, because they are the point:

- **`somewhere peaceful where I can sleep`** scores every review zero. The right answers are invisible.
- **`Hotel du Marais`** is a clean win — exact strings are keyword search's home turf.
- **`a short stroll from the Louvre`** ranks the correct review (Rue de Rivoli, museums) *third*, behind a calm-courtyard review that only shares the word "from."
- **`a hotel not near the airport`** returns the airport review first. Keyword search cannot read the "not."

The root cause is the **vocabulary mismatch problem**: people and documents use different words
for the same idea, and keyword search matches surface forms, not meaning.

### 3.5 Can preprocessing fix it? Stemming and lemmatization

Part of the gap is mechanical: the corpus says *slept*, the query says *sleep*. Classical keyword
search normalizes word forms before indexing. **Stemming** chops suffixes with fixed rules (fast, crude);
**lemmatization** maps to a dictionary form using the part of speech (slower, accurate). Watch where each stops.

In [7]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download("wordnet", quiet=True)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

for word in ["sleeping", "sleeps", "slept"]:
    stem = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos="v")   # pos='v' = treat as a verb
    print(f"{word:10s} stem={stem:8s} lemma={lemma}")

sleeping   stem=sleep    lemma=sleep
sleeps     stem=sleep    lemma=sleep
slept      stem=slept    lemma=sleep


Stemming unifies the regular forms but leaves the irregular *slept* alone; lemmatization fixes it.
Neither touches *quiet* vs *peaceful* — morphology only changes how a word looks, never what it means.
That remaining gap is what semantic search closes.

## 4. Chunking: context limits and retrieval precision

Every encoder accepts a bounded amount of text. The original BERT caps at 512 tokens and
*silently truncates* anything longer. The cell below loads a full hotel page and shows how much
of it BERT never sees.

In [8]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
page = open("hotel_page.txt", encoding="utf-8").read()

full = bert_tokenizer.encode(page, truncation=False)
truncated = bert_tokenizer.encode(page, truncation=True, max_length=512)

print(f"characters in document   : {len(page):,}")
print(f"tokens, no truncation    : {len(full):,}")
print(f"tokens, truncated to 512 : {len(truncated):,}")
print(f"fraction of document seen: {len(truncated)/len(full):.1%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (846 > 512). Running this sequence through the model will result in indexing errors


characters in document   : 4,118
tokens, no truncation    : 846
tokens, truncated to 512 : 512
fraction of document seen: 60.5%


The encoder we use, **nomic-embed-text-v1.5**, accepts 8,192 tokens, so this page fits whole.
But chunking is not only about the hard limit. A single vector for a long document has to average
everything it says, and a vector that means everything matches nothing in particular. We chunk to
keep each vector specific. Here the reviews are one sentence each, so each review is its own chunk.

## 5. Semantic search with nomic-embed-text-v1.5

### 5.1 Loading the encoder and encoding the corpus

`nomic-embed-text-v1.5` produces 768-dimensional vectors and was trained with **task prefixes**:
prepend `search_document: ` to passages and `search_query: ` to queries. Omitting them degrades
retrieval quietly, so this is the single most common mistake with this model.

In [9]:
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

doc_embeddings = model.encode(["search_document: " + review for review in REVIEWS])
print(f"embeddings shape: {doc_embeddings.shape}")   # (8, 768)

<All keys matched successfully>


[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


embeddings shape: (8, 768)


### 5.2 Cosine similarity search

Cosine similarity is 1 when two vectors point the same way, 0 when unrelated. It ignores
vector length and keeps only direction, which is what we want when comparing meaning.

In [10]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def semantic_search(query, doc_embeddings, documents, model):
    query_embedding = model.encode("search_query: " + query)
    scored = [(i, cosine(query_embedding, doc_embeddings[i])) for i in range(len(documents))]
    return sorted(scored, key=lambda x: x[1], reverse=True)


for doc_id, sim in semantic_search("somewhere peaceful where I can sleep",
                                   doc_embeddings, REVIEWS, model)[:3]:
    print(f"[{doc_id}] {sim:.4f}  {REVIEWS[doc_id]}")

[0] 0.5668  We slept with the windows open and never heard a thing, right in the middle of Paris.
[1] 0.5590  A calm little place far from the traffic, with birds singing in the courtyard.
[6] 0.5127  The bar by the pool blasted music until well past midnight.


The windows-open and calm-courtyard reviews, which keyword search scored at zero, come back first
and second, though neither shares a word with the query. That is the entire case for semantic
retrieval. Note the third result, the pool noise complaint scoring close behind: embeddings
measure what a text is *about* (sound at night) and miss its polarity, which is why Chapter 4's
reranker exists.

## 6. Keyword against semantic, side by side

In [11]:
def compare(query):
    print("=" * 64)
    print(f"query: '{query}'")
    print("-" * 64)
    print(" keyword (TF-IDF):")
    for doc_id, score in keyword_search(query, REVIEWS, tokenized_docs, inverted_index)[:3]:
        print(f"   [{doc_id}] {score:7.4f}  {REVIEWS[doc_id]}")
    print(" semantic (nomic):")
    for doc_id, sim in semantic_search(query, doc_embeddings, REVIEWS, model)[:3]:
        print(f"   [{doc_id}] {sim:7.4f}  {REVIEWS[doc_id]}")
    print()


for q in QUERIES:
    compare(q)

query: 'somewhere peaceful where I can sleep'
----------------------------------------------------------------
 keyword (TF-IDF):
   [0]  0.0000  We slept with the windows open and never heard a thing, right in the middle of Paris.
   [1]  0.0000  A calm little place far from the traffic, with birds singing in the courtyard.
   [2]  0.0000  The walls were paper-thin and we heard every footstep in the hallway.
 semantic (nomic):
   [0]  0.5668  We slept with the windows open and never heard a thing, right in the middle of Paris.
   [1]  0.5590  A calm little place far from the traffic, with birds singing in the courtyard.
   [6]  0.5127  The bar by the pool blasted music until well past midnight.

query: 'Hotel du Marais'
----------------------------------------------------------------
 keyword (TF-IDF):
   [3]  4.1589  Hotel du Marais was spotless and the staff upgraded our room for free.
   [0]  0.0000  We slept with the windows open and never heard a thing, right in the middle of Par

## 7. Takeaways

| | Keyword (TF-IDF / BM25) | Semantic (embeddings) |
|---|---|---|
| **Strengths** | Fast, no model, precise on exact strings | Handles paraphrase, synonyms, concepts |
| **Weaknesses** | Blind to paraphrase (vocabulary mismatch) | Fuzzier on exact strings, costs an encoder pass |
| **Best for** | Names, codes, known terminology | Open-ended questions in the user's words |
| **Shared failure** | Negation | Negation |

Keyword and semantic scores are on different scales (unbounded TF-IDF vs. cosine in [0, 1]), so
combining them — *hybrid search* — needs rank fusion or normalization, which Chapter 6 builds
directly on the two retrievers above.